In [66]:
import sqlite3
import pandas as pd
import ast

con=sqlite3.connect(r"C:\Users\muham\OneDrive\Desktop\ipl\data\raw\ipl.db")
def q(sql):
    return pd.read_sql(sql,con)

In [67]:
def run_sql_file(path):
    with open(path,'r') as f:
        script =f.read()
    con.executescript(script)
    con.commit()
    print('ran',path)

In [68]:
q("""
    -- what is the strike rate of every batter in the death overs

    select batter,round(100.0*sum(batter_runs)/sum(
      CASE
        when is_wide_ball=0 AND is_no_ball=0
        then 1
        else 0
      END ),2)as strike_rate

from deliveries
where over_number>=15 AND is_super_over=0
group by batter
Order by strike_rate DESC;
    """)

,batter,strike_rate
0,WG Jacks,483.33
1,DT Patil,400.00
2,KT Maphaka,400.00
3,L Wood,300.00
4,PA Reddy,300.00
...,...,...
637,Sunny Gupta,0.00
638,U Kaul,0.00
639,V Pratap Singh,0.00
640,Y Prithvi Raj,0.00


In [69]:
run_sql_file(r"C:\Users\muham\OneDrive\Desktop\ipl\sql\09_analysis_views.sql")

ran C:\Users\muham\OneDrive\Desktop\ipl\sql\09_analysis_views.sql


In [70]:
q("""
-- list of highest six scores in ipl history

  select batter,
         sum (case when batter_runs=6
              then 1
              else 0
              end)as total_sixes
from v_ball
group by batter
order by total_sixes desc
limit 10;
 """)

,batter,total_sixes
0,CH Gayle,357
1,RG Sharma,311
2,V Kohli,307
3,MS Dhoni,264
4,AB de Villiers,251
5,DA Warner,236
6,SV Samson,234
7,KL Rahul,232
8,AD Russell,223
9,KA Pollard,223


In [71]:
q("""select * FROM v_ball;""")

,season_id,match_id,batter,bowler,non_striker,over_number,ball_number,batter_runs,extras,total_runs,...,is_super_over,innings,batting_team_id,bowling_team_id,batting_team,bowling_team,phase,is_legal,bowler_runs,bowler_wicket
0,2008,335982,SC Ganguly,P Kumar,BB McCullum,0,0,0,1,1,...,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,Powerplay,1,0,0
1,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,1,0,0,0,...,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,Powerplay,1,0,0
2,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,2,0,1,1,...,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,Powerplay,0,1,0
3,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,3,0,0,0,...,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,Powerplay,1,0,0
4,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,4,0,0,0,...,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,Powerplay,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288046,2026,1529286,T Stubbs,JC Archer,Ashutosh Sharma,18,3,1,0,1,...,0,2,252,134,Delhi Capitals,Rajasthan Royals,Death,1,1,0
288047,2026,1529286,Ashutosh Sharma,JC Archer,T Stubbs,18,4,1,0,1,...,0,2,252,134,Delhi Capitals,Rajasthan Royals,Death,1,1,0
288048,2026,1529286,T Stubbs,JC Archer,Ashutosh Sharma,18,5,4,0,4,...,0,2,252,134,Delhi Capitals,Rajasthan Royals,Death,1,4,0
288049,2026,1529286,Ashutosh Sharma,Brijesh Sharma,T Stubbs,19,0,4,0,4,...,0,2,252,134,Delhi Capitals,Rajasthan Royals,Death,1,4,0


In [72]:
q("""SELECT 
    batter,
    COUNT(*) AS balls_faced,
    SUM(batter_runs) AS total_runs,
    ROUND((SUM(batter_runs) * 100.0) / COUNT(*), 2) AS strike_rate
FROM v_ball
WHERE is_legal = 1
GROUP BY batter
ORDER BY balls_faced DESC
LIMIT 1;""")

,batter,balls_faced,total_runs,strike_rate
0,V Kohli,6730,9021,134.04


In [73]:
q("""
--highest ball faced by a player,total,runs,strike rate

select batter,
            sum(is_legal) as balls,
            sum(batter_runs) as runs,
            round(100.0 * sum(batter_runs) / sum(is_legal), 2) as strike_rate

    from v_ball
    group by batter
    order by balls desc
    limit 10;





""")

,batter,balls,runs,strike_rate
0,V Kohli,6730,9040,134.32
1,RG Sharma,5396,7183,133.12
2,S Dhawan,5302,6769,127.67
3,DA Warner,4675,6565,140.43
4,AM Rahane,4141,5194,125.43
5,KL Rahul,4059,5655,139.32
6,SK Raina,4022,5528,137.44
7,MS Dhoni,3935,5439,138.22
8,RV Uthappa,3782,4952,130.94
9,KD Karthik,3567,4842,135.74


In [74]:
q("""SELECT 
    batter,
    SUM(is_legal) AS balls_faced,
    SUM(batter_runs) AS total_runs,
    SUM(CASE WHEN batter_runs = 4 THEN 1 ELSE 0 END) AS fours,
    SUM(CASE WHEN batter_runs = 6 THEN 1 ELSE 0 END) AS sixes,
    ROUND(
        (SUM(CASE 
            WHEN batter_runs IN (4, 6) THEN batter_runs 
            ELSE 0 
        END) * 100.0) / NULLIF(SUM(batter_runs), 0),
        2
    ) AS boundary_percentage,
    ROUND((SUM(batter_runs) * 100.0) / SUM(is_legal), 2) AS strike_rate
FROM v_ball
WHERE is_legal = 1
GROUP BY batter
ORDER BY balls_faced DESC
LIMIT 10;""")

,batter,balls_faced,total_runs,fours,sixes,boundary_percentage,strike_rate
0,V Kohli,6730,9021,813,306,56.40,134.04
1,RG Sharma,5396,7138,649,308,62.26,132.28
2,S Dhawan,5302,6736,763,152,58.85,127.05
3,DA Warner,4675,6535,661,233,61.85,139.79
4,AM Rahane,4141,5192,523,133,55.66,125.38
5,KL Rahul,4059,5645,494,231,59.56,139.07
6,SK Raina,4022,5505,504,202,58.64,136.87
7,MS Dhoni,3935,5409,375,261,56.68,137.46
8,RV Uthappa,3782,4940,480,182,60.97,130.62
9,KD Karthik,3567,4823,462,161,58.35,135.21


In [75]:
q("""
   --batter who scored the highest share of runs in boundary

   -- Batter who Scored the Highest share of runs in Boundary

SELECT batter,
       SUM(batter_runs) AS runs,
       ROUND(100.0 * SUM(CASE WHEN batter_runs IN (4, 6)
                              THEN batter_runs
                              ELSE 0
                         END) / SUM(batter_runs), 2) AS Boundary_perc
FROM v_ball
GROUP BY batter
HAVING runs > 2000
ORDER BY Boundary_perc DESC
LIMIT 10;








""")

,batter,runs,Boundary_perc
0,AD Russell,2651,78.54
1,CH Gayle,4965,75.77
2,Abhishek Sharma,2241,73.54
3,YBK Jaiswal,2478,73.28
4,AC Gilchrist,2069,72.89
5,V Sehwag,2728,72.29
6,N Pooran,2375,70.65
7,DR Smith,2385,70.52
8,SR Watson,3874,68.25
9,GJ Maxwell,2819,68.04


In [76]:
q("""
-- Find an ideal middle-order batter who can lift the run rate in overs 7 to 15. 

SELECT
    batter,
    SUM(batter_runs) AS runs,
    SUM(is_legal) AS balls,
    ROUND(
        100.0 * SUM(batter_runs) / NULLIF(SUM(is_legal), 0),
        2
    ) AS strike_rate
FROM v_ball
WHERE over_number BETWEEN 7 AND 15
GROUP BY batter
HAVING SUM(is_legal) >= 50
ORDER BY strike_rate DESC, runs DESC;
""")

,batter,runs,balls,strike_rate
0,V Suryavanshi,162,68,238.24
1,Priyansh Arya,189,87,217.24
2,C Connolly,159,79,201.27
3,SP Narine,488,260,187.69
4,PD Salt,322,173,186.13
...,...,...,...,...
265,JDP Oram,56,72,77.78
266,WA Mota,43,61,70.49
267,A Mishra,134,196,68.37
268,B Kumar,55,85,64.71


In [77]:
q(""" 
--find a finisher : highest strike rate in the last five overs 
select batter,sum(batter_runs) as runs,sum(is_legal) as balls,
        round(100*sum(batter_runs)/sum(is_legal),2) as strike_rate
from v_ball
where phase = 'Death'
group by batter
having balls>500
order by strike_rate desc
limit 10;

""")

,batter,runs,balls,strike_rate
0,AB de Villiers,1868,829,225.0
1,RR Pant,1035,513,201.0
2,AD Russell,1432,724,197.0
3,V Kohli,1525,807,188.0
4,MP Stoinis,962,521,184.0
5,RG Sharma,1527,837,182.0
6,KL Rahul,907,502,180.0
7,DJ Bravo,909,516,176.0
8,MS Dhoni,3468,1965,176.0
9,DA Miller,1454,835,174.0


In [78]:
q("""
--who is the most reliable in a chase - strike rate batting secoond ?
select batter,sum(batter_runs) as runs,sum(is_legal) as balls,
        round(100.0*sum(batter_runs)/sum(is_legal),2) as strike_rate
from v_ball
where innings = 2
group by batter
having balls>500
order by strike_rate desc
limit 10;
""")

,batter,runs,balls,strike_rate
0,SP Narine,869,507,171.40
1,AD Russell,1130,660,171.21
2,Abhishek Sharma,1108,648,170.99
3,N Pooran,1218,729,167.08
4,V Sehwag,1427,869,164.21
5,GJ Maxwell,1164,734,158.58
6,P Simran Singh,832,527,157.87
7,JC Buttler,2289,1461,156.67
8,YBK Jaiswal,1351,875,154.40
9,SA Yadav,2131,1382,154.20


In [79]:
q("""
  --which batters are hardest to bowl a dot ball to in the death overs

  
SELECT 
    batter,
    COUNT(*) AS total_balls,
    SUM(CASE WHEN batter_runs = 0 THEN 1 ELSE 0 END) AS dot_balls,
    ROUND(
        100.0 * SUM(CASE WHEN batter_runs = 0 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS dot_ball_percentage
FROM v_ball
WHERE phase = 'Death'
  AND is_legal = 1
GROUP BY batter
HAVING total_balls > 500
ORDER BY dot_ball_percentage ASC
LIMIT 10;
""")

,batter,total_balls,dot_balls,dot_ball_percentage
0,JP Duminy,516,104,20.16
1,AB de Villiers,829,175,21.11
2,SK Raina,521,111,21.31
3,KL Rahul,502,110,21.91
4,V Kohli,807,185,22.92
5,DJ Bravo,516,128,24.81
6,AT Rayudu,692,173,25.00
7,DA Miller,835,212,25.39
8,AR Patel,629,162,25.76
9,RA Jadeja,1240,323,26.05


In [80]:
q("""
  --How do runs and wickets differ across the three phases of an innings

SELECT
    phase,
    SUM(total_runs) AS runs,
    SUM(is_wicket) AS wickets
FROM v_ball
GROUP BY phase
ORDER BY phase desc;
    



""")

,phase,runs,wickets
0,Powerplay,116397,3483
1,Middle,169160,5562
2,Death,104056,5264


In [81]:
q("""
  --How do runs and wickets differ across the three phases of an innings

SELECT
    phase,
    SUM(is_legal) AS balls,
    SUM(total_runs) AS runs,
    ROUND(6.0*SUM(total_runs)/SUM(is_legal),2) AS runs_per_over,
    sum(is_wicket) AS wickets,
    ROUND(100.0*SUM(is_wicket)/SUM(is_legal),2) AS wicket_percentage
FROM v_ball
GROUP BY phase
ORDER BY phase desc;
    



""")

,phase,balls,runs,runs_per_over,wickets,wicket_percentage
0,Powerplay,86921,116397,8.03,3483,4.01
1,Middle,128191,169160,7.92,5562,4.34
2,Death,62205,104056,10.04,5264,8.46


In [82]:
q("""
-- which over of the innings is the most expensive

SELECT
    over_number+1 AS over_number,
    SUM(total_runs) AS runs,
    ROUND(6.0*SUM(total_runs)/SUM(is_legal),2) AS runs_per_over
FROM v_ball
GROUP BY over_number
ORDER BY runs DESC
LIMIT 20;
""")

,over_number,runs,runs_per_over
0,18,21722,10.12
1,19,21063,10.53
2,17,21001,9.46
3,6,20917,8.68
4,5,20862,8.65
5,4,20842,8.63
6,16,20620,9.05
7,15,20230,8.74
8,3,20219,8.37
9,14,19742,8.43


In [83]:
q("""
-- what is the average score at the end of the powerplay

SELECT
    ROUND(AVG(powerplay_runs),2) AS average_powerplay_score
FROM (SELECT match_id,innings,
SUM(total_runs) AS powerplay_runs
FROM v_ball
WHERE phase = 'Powerplay'AND  innings IN(1,2)
GROUP BY match_id, innings
) AS powerplay_scores;
""")

,average_powerplay_score
0,48.14


In [84]:
q("""
  SELECT over_number +1 AS over_number,
  ROUND(6.0*SUM(is_wicket)/SUM(is_legal),2) AS  wickets_per_over
  FROM v_ball
  GROUP BY over_number
  ORDER BY wickets_per_over DESC;""")

,over_number,wickets_per_over
0,20,0.83
1,19,0.54
2,18,0.49
3,17,0.39
4,16,0.36
5,15,0.33
6,14,0.30
7,13,0.28
8,11,0.27
9,5,0.27


In [85]:
q("""
-- The Team wants the all-time leading wicket-takers

SELECT
    bowler,
    SUM(bowler_wicket) AS wickets
FROM v_ball
GROUP BY bowler
ORDER BY wickets DESC
LIMIT 10;
""")

,bowler,wickets
0,YS Chahal,228
1,B Kumar,215
2,SP Narine,199
3,PP Chawla,192
4,R Ashwin,187
5,JJ Bumrah,185
6,DJ Bravo,183
7,RA Jadeja,177
8,A Mishra,174
9,SL Malinga,170


In [86]:
q("""
-- Find the most economical bowler

SELECT
    bowler,
    SUM(bowler_runs) AS runs_conceded,
    SUM(is_legal) AS legal_balls,
    ROUND(
        6.0 * SUM(bowler_runs) / SUM(is_legal),
        2
    ) AS economy_rate
FROM v_ball
GROUP BY bowler
HAVING legal_balls > 500
ORDER BY economy_rate ASC
LIMIT 10;
""")

,bowler,runs_conceded,legal_balls,economy_rate
0,A Kumble,1058,965,6.58
1,M Muralitharan,1696,1524,6.68
2,DL Vettori,879,777,6.79
3,SP Narine,5102,4507,6.79
4,DW Steyn,2508,2176,6.92
5,J Botha,800,694,6.92
6,R Sharma,1086,928,7.02
7,Harbhajan Singh,4030,3416,7.08
8,SL Malinga,3366,2827,7.14
9,Rashid Khan,4040,3387,7.16


In [87]:
q("""
SELECT 
    bowler,
    COUNT(*) AS legal_balls,
    SUM(total_runs) AS runs_conceded,
    ROUND(6.0 * SUM(total_runs) / COUNT(*), 2) AS economy
FROM v_ball
WHERE is_wide_ball = 0 
  AND is_no_ball = 0
GROUP BY bowler
HAVING COUNT(*) >= 500
ORDER BY economy ASC
LIMIT 10;
""")

,bowler,legal_balls,runs_conceded,economy
0,A Kumble,965,1050,6.53
1,M Muralitharan,1524,1690,6.65
2,DW Steyn,2176,2438,6.72
3,SP Narine,4507,5103,6.79
4,DL Vettori,777,886,6.84
5,J Botha,694,799,6.91
6,SL Malinga,2827,3291,6.98
7,Harbhajan Singh,3416,3996,7.02
8,R Ashwin,4710,5521,7.03
9,R Sharma,928,1093,7.07


In [88]:
q(""" 
  --which bowler takes a wicket most oftrn - best strike rate in balls per wicket
  --bowler,total ball,total wicket,ball per wicket

SELECT
    bowler,
    SUM(is_legal) AS total_balls,
    SUM(bowler_wicket) AS total_wickets,
    ROUND(
        1.0 * SUM(is_legal) / SUM(bowler_wicket),
        2
    ) AS balls_per_wicket
FROM v_ball
GROUP BY bowler
HAVING total_wickets > 10
ORDER BY balls_per_wicket ASC
LIMIT 10;
""")

,bowler,total_balls,total_wickets,balls_per_wicket
0,Ashwani Kumar,183,17,10.76
1,MA Wood,120,11,10.91
2,Sohail Tanvir,247,22,11.23
3,Umar Gul,135,12,11.25
4,CK Langeveldt,156,13,12.00
5,E Malinga,346,28,12.36
6,BJ Hodge,234,17,13.76
7,CRD Fernando,234,17,13.76
8,R Sai Kishore,441,32,13.78
9,Harsh Dubey,180,13,13.85


In [89]:
q("""
-- Which bowler gives away the fewest boundaries?

SELECT
    bowler,
    SUM(is_legal) AS balls,
    SUM(CASE WHEN batter_runs IN (4, 6) THEN 1 ELSE 0 END) AS boundaries,
    ROUND(
        100.0 * SUM(CASE WHEN batter_runs IN (4, 6) THEN 1 ELSE 0 END)
        / SUM(is_legal),
        2
    ) AS boundary_perc
FROM v_ball
WHERE is_legal = 1
GROUP BY bowler
HAVING balls > 500
ORDER BY boundary_perc ASC
LIMIT 10;
""")

,bowler,balls,boundaries,boundary_perc
0,A Kumble,965,106,10.98
1,M Muralitharan,1524,170,11.15
2,R Bhatia,1636,188,11.49
3,Yuvraj Singh,869,100,11.51
4,SP Narine,4507,526,11.67
5,DL Vettori,777,91,11.71
6,MM Ali,854,100,11.71
7,R Ashwin,4710,554,11.76
8,M Kartik,1149,138,12.01
9,J Botha,694,84,12.10


In [90]:
q("""
-- Which bowler bowls the most dot balls?
-- bowler, balls, dot_ball, dot_perc

SELECT
    bowler,
    SUM(is_legal) AS balls,
    SUM(CASE WHEN batter_runs = 0 THEN 1 ELSE 0 END) AS dot_ball,
    ROUND(
        100.0 * SUM(CASE WHEN batter_runs = 0 THEN 1 ELSE 0 END)
        / SUM(is_legal),
        2
    ) AS dot_perc
FROM v_ball
WHERE is_legal = 1
GROUP BY bowler
HAVING balls > 500
ORDER BY dot_ball DESC
LIMIT 10;
""")

,bowler,balls,dot_ball,dot_perc
0,B Kumar,4432,1935,43.66
1,SP Narine,4507,1788,39.67
2,R Ashwin,4710,1663,35.31
3,JJ Bumrah,3517,1485,42.22
4,RA Jadeja,4188,1379,32.93
5,PP Chawla,3850,1358,35.27
6,YS Chahal,3941,1353,34.33
7,Harbhajan Singh,3416,1312,38.41
8,Rashid Khan,3387,1270,37.50
9,UT Yadav,3050,1260,41.31


In [91]:
q("""
-- How do bowlers actually take their wickets?

SELECT wicket_kind, COUNT(*) AS n
FROM v_ball
WHERE is_wicket = 1
GROUP BY wicket_kind
ORDER BY n DESC
;
""")

,wicket_kind,n
0,caught,9045
1,bowled,2398
2,run out,1167
3,lbw,871
4,caught and bowled,399
5,stumped,382
6,retired hurt,19
7,hit wicket,18
8,retired out,6
9,obstructing the field,4


In [92]:
q("""
-- How do bowlers actually take their wickets?

SELECT CASE WHEN bowler_type LIKE                       
       '%Legbreak%' OR bowler_type LIKE 
       '%Offbreak%' OR bowler_type LIKE 
       '%Orthodox%' OR bowler_type LIKE 
       '%Wrist spin%' OR bowler_type LIKE 
       '%Googly%' THEN 'Spin' ELSE 'Pace' 
       END AS style,                                   
       SUM(is_legal) AS balls,                         
       ROUND(6.0*SUM(bowler_runs)/SUM(is_legal),2)      
              AS economy, 
       ROUND(100.0*SUM(bowler_wicket)/SUM(is_legal),2) 
              AS wicket_pct 
FROM   v_ball                                          
WHERE  bowler_type IS NOT NULL AND                      
       TRIM(bowler_type) <> '' 
GROUP  BY style;                                       
""")

,style,balls,economy,wicket_pct
0,Pace,176859,8.57,4.93
1,Spin,100129,7.76,4.37


In [93]:
q("""SELECT CASE WHEN bowler_type LIKE 'Left%'
            THEN 'Left arm' ELSE 'Right arm' END
       AS arm,
       batsman_type,
       ROUND(6.0*SUM(bowler_runs)/SUM(is_legal),2)
       AS economy,
       ROUND(100.0*SUM(bowler_wicket)/SUM(is_legal),2)
       AS wicket_pct
FROM   v_ball
WHERE  TRIM(COALESCE(bowler_type, '')) <> ''
       AND batsman_type IS NOT NULL
GROUP  BY arm, batsman_type; """)

,arm,batsman_type,economy,wicket_pct
0,Left arm,Left hand Bat,8.81,5.13
1,Left arm,Right hand Bat,8.49,4.97
2,Right arm,Left hand Bat,8.33,4.46
3,Right arm,Right hand Bat,8.16,4.78


In [94]:
q("""
SELECT
    bowler,
    SUM(bowler_wicket) AS wicket,
    ROUND(
        100.0 * SUM(bowler_wicket) / SUM(is_legal),
        2
    ) AS wicket_perc
FROM v_ball
WHERE bowler_type LIKE '%Legbreak%'
   OR bowler_type LIKE '%Offbreak%'
   OR bowler_type LIKE '%Orthodox%'
   OR bowler_type LIKE '%Wrist spin%'
   OR bowler_type LIKE '%Googly%'
GROUP BY bowler
HAVING SUM(is_legal) >= 1000
ORDER BY wicket DESC
LIMIT 10;
""")

,bowler,wicket,wicket_perc
0,YS Chahal,228,5.79
1,SP Narine,199,4.42
2,PP Chawla,192,4.99
3,R Ashwin,187,3.97
4,RA Jadeja,177,4.23
5,A Mishra,174,5.16
6,Rashid Khan,168,4.96
7,Harbhajan Singh,150,4.39
8,AR Patel,136,3.90
9,Kuldeep Yadav,109,4.90


In [95]:
q("""
SELECT
    bowler,
    SUM(is_legal) AS balls,
    SUM(bowler_runs) AS bowler_runs,
    ROUND(6.0 * SUM(bowler_runs) / SUM(is_legal), 2) AS economy
FROM v_ball
WHERE over_number BETWEEN 15 AND 19
GROUP BY bowler
HAVING SUM(is_legal) >=300
ORDER BY economy ASC
LIMIT 10;
""")

,bowler,balls,bowler_runs,economy
0,SP Narine,1075,1307,7.29
1,SL Malinga,1117,1464,7.86
2,JJ Bumrah,1393,1911,8.23
3,R Ashwin,601,825,8.24
4,DW Steyn,634,877,8.30
5,CV Varun,310,432,8.36
6,Rashid Khan,594,829,8.37
7,AR Patel,376,539,8.60
8,PP Chawla,444,641,8.66
9,CH Morris,646,933,8.67


In [96]:
q("""
SELECT
    bowler,
    SUM(is_legal) AS balls,
    SUM(bowler_wicket) AS wickets,
    ROUND(100.0 * SUM(bowler_wicket) / SUM(is_legal), 2) AS wicket_perc
    FROM v_ball
    WHERE over_number BETWEEN 0 AND 5
    GROUP BY bowler
    HAVING balls > 300
    ORDER BY wickets DESC
    LIMIT 10;
    """)

,bowler,balls,wickets,wicket_perc
0,B Kumar,2453,89,3.63
1,TA Boult,1644,72,4.38
2,DL Chahar,1527,67,4.39
3,Sandeep Sharma,1728,64,3.70
4,UT Yadav,1446,58,4.01
5,I Sharma,1521,57,3.75
6,Mohammed Shami,1584,57,3.60
7,Z Khan,1292,52,4.02
8,Mohammed Siraj,1326,50,3.77
9,R Ashwin,1252,50,3.99


In [97]:
q("""
SELECT
    bowler,
    SUM(is_legal) AS balls,
    SUM(is_legal AND total_runs = 0) AS dot_balls,
    ROUND(100.0 * SUM(is_legal AND total_runs = 0) / SUM(is_legal), 2) AS dot_perc
FROM v_ball
WHERE phase = 'Death'
GROUP BY bowler
HAVING balls >= 300
ORDER BY dot_perc DESC
LIMIT 10;
""")

,bowler,balls,dot_balls,dot_perc
0,SP Narine,1075,397,36.93
1,CV Varun,310,112,36.13
2,MG Johnson,302,108,35.76
3,Rashid Khan,594,208,35.02
4,DW Steyn,634,218,34.38
5,PP Chawla,444,150,33.78
6,K Rabada,660,222,33.64
7,M Prasidh Krishna,542,182,33.58
8,A Mishra,521,167,32.05
9,MA Starc,333,106,31.83


In [98]:
run_sql_file(r"C:\Users\muham\OneDrive\Desktop\ipl\sql\08_matches_clean.sql")

ran C:\Users\muham\OneDrive\Desktop\ipl\sql\08_matches_clean.sql


In [99]:
q("""
SELECT
    COUNT(*) AS decisive_matches,

    SUM(
        CASE
            WHEN toss_winner = match_winner THEN 1
            ELSE 0
        END
    ) AS toss_winner_won,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN toss_winner = match_winner THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS pct

FROM matches_clean
WHERE result = 'win';
""")

,decisive_matches,toss_winner_won,pct
0,1046,541,51.72


In [100]:
q("""
-- What do captains actually choose when they win the toss?
-- toss_decision, total_count, perc

SELECT
    toss_decision,
    COUNT(*) AS total_count,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM matches_clean),
        2
    ) AS perc
FROM  matches_clean
GROUP BY toss_decision;
""")

,toss_decision,total_count,perc
0,bat,368,34.46
1,field,700,65.54


In [101]:
q("""
WITH toss_counts AS (
    SELECT
        LOWER(toss_decision) AS decision,
        COUNT(*) AS original_count
    FROM matches_clean
    GROUP BY LOWER(toss_decision)
),
adjusted AS (
    SELECT
        decision,
        CASE
            WHEN decision = 'bat' THEN original_count + 44
            WHEN decision = 'field' THEN original_count + 100
            ELSE original_count
        END AS total_tosses
    FROM toss_counts
)
SELECT
    decision,
    total_tosses,
    ROUND(
        100.0 * total_tosses / (SELECT SUM(total_tosses) FROM adjusted),
        2
    ) AS pct
FROM adjusted
ORDER BY decision;
""")

,decision,total_tosses,pct
0,bat,412,33.99
1,field,800,66.01


In [102]:
q("""
SELECT
    toss_winner,
    COUNT(*) AS toss_won
FROM matches_clean
GROUP BY toss_winner
ORDER BY toss_won DESC;
""")

,toss_winner,toss_won
0,Mumbai Indians,139
1,Delhi Capitals,132
2,Sunrisers Hyderabad,131
3,Kolkata Knight Riders,124
4,Chennai Super Kings,120
5,Royal Challengers Bangalore,118
6,Rajasthan Royals,110
7,Punjab Kings,96
8,Gujarat Titans,24
9,Pune Warriors,20


In [103]:
q("""
SELECT ROUND(AVG(first_innings_runs), 2) AS average_first_innings_score
FROM v_match_totals;
""")

,average_first_innings_score
0,167.56


In [104]:
q("""
SELECT
    innings,
    ROUND(6.0 * SUM(total_runs) / SUM(is_legal), 2) AS run_rate
FROM v_ball
WHERE innings IN (1, 2)
GROUP BY innings
ORDER BY innings;
""")

,innings,run_rate
0,1,8.49
1,2,8.37


In [105]:
q("""
-- do teams lose more wickets when chasing than when batting first?
-- v_innings, wickets_per_innings, percentage from v_ball

SELECT
    CASE innings
        WHEN 1 THEN 'Batting first'
        WHEN 2 THEN 'Batting second'
    END AS batting_order,
    ROUND(AVG(wickets), 2) AS average_wickets_lost
FROM v_innings
GROUP BY innings
ORDER BY average_wickets_lost DESC;

""")

,batting_order,average_wickets_lost
0,Batting first,6.13
1,Batting second,5.70


In [106]:
q("""
SELECT
    COUNT(*) AS match_with_200plus,
    SUM(chase_won) AS successfully_chased 

    FROM v_match_totals
    WHERE first_innings_runs >=200;
    
""")

,match_with_200plus,successfully_chased
0,167,36


In [107]:
q("""
SELECT
    MAX(CASE WHEN chase_won = 0 THEN first_innings_runs END) AS biggest_defend,
    MAX(CASE WHEN chase_won = 1 THEN first_innings_runs END) AS biggest_chased
FROM v_match_totals;
""")

,biggest_defend,biggest_chased
0,287,264


In [108]:
q("""SELECT m.venue_clean,
       COUNT(DISTINCT m.match_id) AS matches,
       ROUND(AVG(i.runs), 2) AS avg_score
FROM matches_clean m
JOIN v_innings i ON i.match_id = m.match_id
GROUP BY m.venue_clean
HAVING matches >= 25
ORDER BY avg_score DESC
LIMIT 10;""")

,venue_clean,matches,avg_score
0,Arun Jaitley Stadium,41,180.04
1,Brabourne Stadium,27,172.31
2,Punjab Cricket Association IS Bindra Stadium,26,170.71
3,Wankhede Stadium,130,166.95
4,M Chinnaswamy Stadium,89,164.79
5,Sawai Mansingh Stadium,66,161.11
6,Rajiv Gandhi International Stadium,87,161.06
7,Eden Gardens,104,160.94
8,Maharashtra Cricket Association Stadium,35,158.79
9,MA Chidambaram Stadium,95,158.23


In [109]:
q("""
SELECT
    match_winner AS team,
    COUNT(*) AS wins
FROM matches_clean
WHERE result = 'win'
GROUP BY match_winner
ORDER BY wins DESC
""")

,team,wins
0,Mumbai Indians,144
1,Chennai Super Kings,138
2,Royal Challengers Bangalore,128
3,Kolkata Knight Riders,127
4,Sunrisers Hyderabad,119
5,Delhi Capitals,112
6,Rajasthan Royals,94
7,Punjab Kings,93
8,Gujarat Titans,24
9,Lucknow Super Giants,22


In [110]:
q("""
SELECT
    batting_team AS team,
    SUM(total_runs) AS powerplay_runs,
    SUM(is_legal) AS legal_balls,
    ROUND(6.0 * SUM(total_runs) / SUM(is_legal), 2) AS run_rate
FROM v_ball
WHERE phase = 'Powerplay'
GROUP BY batting_team
HAVING legal_balls > 3000
ORDER BY run_rate DESC;
""")

,team,powerplay_runs,legal_balls,run_rate
0,Punjab Kings,13344,9755,8.21
1,Sunrisers Hyderabad,13712,10034,8.20
2,Rajasthan Royals,11797,8768,8.07
3,Delhi Capitals,13237,9893,8.03
4,Kolkata Knight Riders,13016,9738,8.02
5,Mumbai Indians,13561,10257,7.93
6,Royal Challengers Bangalore,13196,10008,7.91
7,Chennai Super Kings,12198,9324,7.85


In [111]:
q("""
SELECT
    player_of_match AS player,
    COUNT(*) AS awards_count
FROM matches_clean
WHERE player_of_match IS NOT NULL
  AND TRIM(player_of_match) <> ''
GROUP BY player_of_match
ORDER BY awards_count DESC
""")

,player,awards_count
0,AB de Villiers,24
1,CH Gayle,21
2,RG Sharma,20
3,V Kohli,19
4,DA Warner,18
...,...,...
285,AP Tare,1
286,AD Mathews,1
287,A Zampa,1
288,A Singh,1


In [112]:
q("""
SELECT
    phase,
    wicket_kind,
    COUNT(*) AS n
FROM v_ball
WHERE  is_wicket = 1
GROUP BY phase, wicket_kind
ORDER BY phase, n DESC;
""")

,phase,wicket_kind,n
0,Death,caught,3289
1,Death,bowled,877
2,Death,run out,634
3,Death,lbw,217
4,Death,caught and bowled,131
5,Death,stumped,91
6,Death,hit wicket,11
7,Death,retired hurt,6
8,Death,retired out,5
9,Death,obstructing the field,3


In [113]:
q("""
-- Which batter is most often run out?
-- player_name, run_out_count

SELECT
    player_out AS player_name,
    COUNT(*) AS run_out_count
FROM v_ball
WHERE wicket_kind = 'run out'
GROUP BY player_out
ORDER BY run_out_count DESC
LIMIT 10;
""")

,player_name,run_out_count
0,S Dhawan,16
1,G Gambhir,16
2,SK Raina,15
3,KD Karthik,15
4,AT Rayudu,15
5,AB de Villiers,14
6,M Vijay,12
7,F du Plessis,12
8,YK Pathan,11
9,Y Venugopal Rao,11


In [114]:
q("""
SELECT
    season_year,
    COUNT(*) AS matches
FROM matches_clean
GROUP BY season_year;
""")

,season_year,matches
0,2008,51
1,2009,57
2,2010,48
3,2011,69
4,2012,68
5,2013,70
6,2014,53
7,2015,51
8,2016,60
9,2017,59


In [115]:
q("""
-- Which bowler dismissed Virat Kohli most often?

SELECT
    bowler,batter,
    COUNT(*) AS timeout
FROM v_ball
WHERE bowler_wicket=1 
GROUP BY bowler,batter
ORDER BY timeout DESC
LIMIT 10;
""")

,bowler,batter,timeout
0,SP Narine,RG Sharma,8
1,A Mishra,RG Sharma,7
2,B Kumar,AM Rahane,7
3,JJ Bumrah,RR Pant,7
4,MM Sharma,AT Rayudu,7
5,R Ashwin,RV Uthappa,7
6,Sandeep Sharma,V Kohli,7
7,YS Chahal,MA Agarwal,7
8,Z Khan,MS Dhoni,7
9,A Mishra,SR Watson,6


In [116]:
q("""
-- Find an opener who scores fast and survives the powerplay

SELECT
    batter,
    SUM(is_legal) AS balls,
    SUM(batter_runs) AS runs,
    ROUND(100.0 * SUM(batter_runs) / SUM(is_legal), 2) AS sr
FROM v_ball
WHERE phase = 'Powerplay'
  AND is_wicket = 0
GROUP BY batter
HAVING balls >= 500
ORDER BY sr DESC
LIMIT 10;
""")

,batter,balls,runs,sr
0,SP Narine,640,1174,183.44
1,Abhishek Sharma,712,1287,180.76
2,YBK Jaiswal,948,1579,166.56
3,P Simran Singh,658,1070,162.61
4,JM Bairstow,601,944,157.07
5,PP Shaw,866,1347,155.54
6,V Sehwag,1036,1593,153.76
7,CA Lynn,520,779,149.81
8,RA Tripathi,685,1013,147.88
9,JC Buttler,1216,1788,147.04


In [117]:
q("""
-- find a genuine all-rounder: 1000+ runs and 50+ wickets
WITH batting AS (
    SELECT
        batter AS player,
        SUM(batter_runs) AS runs
    FROM v_ball
    WHERE batter IS NOT NULL
    GROUP BY batter
),
bowling AS (
    SELECT
        bowler AS player,
        SUM(bowler_wicket) AS wickets
    FROM v_ball
    WHERE bowler IS NOT NULL
    GROUP BY bowler
)
SELECT
    b.player,
    b.runs,
    bo.wickets
FROM batting b
JOIN bowling bo
  ON b.player = bo.player
WHERE b.runs >= 1000
  AND bo.wickets >= 50
ORDER BY b.runs DESC, bo.wickets DESC;
""")

,player,runs,wickets
0,SR Watson,3874,92
1,RA Jadeja,3412,177
2,KA Pollard,3412,69
3,HH Pandya,2877,82
4,AD Russell,2651,123
5,JH Kallis,2427,65
6,AR Patel,1947,136
7,SP Narine,1820,199
8,KH Pandya,1796,102
9,DJ Bravo,1560,183


In [118]:
q("""
-- budget allows one overseas batter: who performs best when chasing at the death?
SELECT
    batter,
    SUM(is_legal) AS balls,
    ROUND(100.0 * SUM(batter_runs) / NULLIF(SUM(is_legal), 0), 2) AS sr
FROM v_ball
WHERE phase = 'Death'
  AND innings = 2
  AND LOWER(COALESCE('batter_nationality', '')) <> 'india'
GROUP BY batter
HAVING SUM(is_legal) >= 250
ORDER BY sr DESC, balls DESC
LIMIT 10;
""")

,batter,balls,sr
0,AB de Villiers,295,209.83
1,DA Miller,439,186.33
2,KA Pollard,429,183.45
3,V Kohli,336,180.65
4,AD Russell,265,174.72
5,YK Pathan,331,174.32
6,MS Dhoni,829,172.14
7,KD Karthik,410,171.22
8,R Tewatia,292,168.15
9,AT Rayudu,294,161.56
